In [1]:
#Use this script to generate train/dev/test splits

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the formatted data
df = pd.read_csv("prelim2.csv")

# Features (X) and labels (y)
X = df[["TokenID", "Exp", "Res", "Univ", "DTAM", "Evid", "TAML", "NegPol", "akt_activity", "akt_accomp", "akt_achiev", "akt_state", "akt_semel"]]  # adjust as needed; taken out for now: , "ant_complete", "ant_initial", "ant_none"
y = df[["eng_constr", "deu_constr", "cmn_constr"]] #taken out for now: , "kat_constr", "yor_constr"

#Drop labels that occur less than twice
#label_counts = y.value_counts()
#y_filtered = y[y.isin(label_counts[label_counts >= 2].index)]
#X_filtered = X.loc[y_filtered.index]

# Drop rows with missing labels first
df = df.dropna(subset=["eng_constr", "deu_constr", "cmn_constr"]) #taken out for now: , "kat_constr", "yor_constr"

# Filter label classes that occur fewer than twice
y = df[["eng_constr", "deu_constr", "cmn_constr"]] #taken out for now: , "kat_constr", "yor_constr"
X = df[["Exp", "Res", "Univ", "DTAM", "Evid", "TAML", "NegPol",
        "akt_activity", "akt_accomp", "akt_achiev", "akt_state", "akt_semel"]] #taken out for now: , "ant_complete", "ant_initial", "ant_none"

label_counts = y.value_counts()
valid_labels = label_counts[label_counts >= 2].index

# Keep only rows with valid multi-labels
mask = y.apply(tuple, axis=1).isin(valid_labels)
X_filtered = X[mask].dropna()
y_filtered = y[mask].dropna()

# Proceed with splitting -- 50/25/25
#X_temp, X_test, y_temp, y_test = train_test_split(X_filtered, y_filtered, test_size=0.2, random_state=42)
#X_train, X_dev, y_train, y_dev = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Proceed with splitting -- 50/50
X_train, X_dev, y_train, y_dev = train_test_split(X, y, test_size=0.5, random_state=42)

In [3]:
print(len(df))

300


In [4]:
#For train/dev/test
#print(f"Train: {len(X_train)} rows")
#print(f"Dev: {len(X_dev)} rows")
#print(f"Test: {len(X_test)} rows")

#For train/dev only
print(f"Train: {len(X_train)} rows")
print(f"Dev: {len(X_dev)} rows")

Train: 150 rows
Dev: 150 rows


In [5]:
print("Train label distribution:")
print(y_train.value_counts(normalize=True))
print("\nDev label distribution:")
print(y_dev.value_counts(normalize=True))
#Include the following for train/dev/test split
#print("\nTest label distribution:")
#print(y_test.value_counts(normalize=True))

Train label distribution:
eng_constr  deu_constr   cmn_constr
perf        perf         le            0.153333
                         n_a           0.140000
                         suo_de        0.066667
                         unmarked      0.060000
pastperf    pastperf     n_a           0.046667
perf        perf         meiyou_guo    0.040000
pastperf    pastperf     unmarked      0.040000
                         suo_de        0.033333
perf        perf         yijing_le     0.026667
pastperf    pastperf     le            0.026667
            perf         le            0.020000
perf        perf         guo           0.020000
                         yijing        0.020000
pastperf    k1_perf      le            0.013333
perf        perf         meiyou        0.013333
            n_a          n_a           0.013333
                         le            0.013333
            perf         yi            0.013333
wouldperf   n_a          n_a           0.013333
pastperf    n_a          n

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

# Train the classifier
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Predict on dev set
y_pred = clf.predict(X_dev)

# Loop through each language column
for i, lang in enumerate(y_dev.columns):
    print(f"\n=== Classification Report for {lang} ===")
    
    # Extract true values
    true_vals = y_dev[lang]
    
    # Step 1: Build mask to exclude 'n_a' and 'ip'
    h = ~true_vals.isin(['n_a', 'ip'])  # "hits": keep only rows not in this set

    # Step 2: Filter X_dev and y_dev for valid rows
    X_dev_loc = X_dev[h]
    true_vals = true_vals[h]
    
    # Step 3: Predict just for these rows
    pred_vals = clf.predict(X_dev_loc)[:, i]

    # Step 4: Report
    print(classification_report(true_vals, pred_vals, zero_division=0))


=== Classification Report for eng_constr ===
              precision    recall  f1-score   support

    condperf       0.00      0.00      0.00         5
   mightperf       0.00      0.00      0.00         1
    pastperf       0.74      0.88      0.81        26
        perf       0.98      0.98      0.98       117
   wouldperf       0.00      0.00      0.00         1

    accuracy                           0.92       150
   macro avg       0.34      0.37      0.36       150
weighted avg       0.90      0.92      0.91       150


=== Classification Report for deu_constr ===
              precision    recall  f1-score   support

     k1_perf       0.00      0.00      0.00         3
  k2_infperf       0.00      0.00      0.00         1
     k2_past       0.40      0.50      0.44         4
 k2_pastperf       0.00      0.00      0.00        10
         n_a       0.00      0.00      0.00         0
        past       0.00      0.00      0.00         8
    pastperf       0.33      0.75      0

In [7]:
from sklearn.metrics import confusion_matrix

for i, lang in enumerate(y_dev.columns):
    print(f"\n=== Confusion Matrix for {lang} ===")
    true_vals = y_dev[lang]
    pred_vals = y_pred[:, i]
    print(confusion_matrix(true_vals, pred_vals))


=== Confusion Matrix for eng_constr ===
[[  0   0   4   0   1]
 [  0   0   1   0   0]
 [  0   0  23   2   1]
 [  0   0   2 115   0]
 [  0   0   1   0   0]]

=== Confusion Matrix for deu_constr ===
[[ 0  0  0  0  0  0  2  1  0]
 [ 0  0  0  0  0  0  1  0  0]
 [ 0  0  2  0  0  0  0  0  2]
 [ 0  0  2  0  0  0  7  1  0]
 [ 0  0  0  0  0  0  1 10  0]
 [ 0  0  1  0  0  0  4  3  0]
 [ 0  0  0  0  0  0  9  2  1]
 [ 0  0  0  0  1  0  4 89  1]
 [ 0  0  0  0  0  0  0  4  2]]

=== Confusion Matrix for cmn_constr ===
[[ 0  2  0  0  3  0  1  1  0  0  0  0  0  0  0]
 [ 0 35  0  1  9  0  0  7  0  0  1  0  0  0  0]
 [ 0  1  0  1  1  0  0  1  0  0  0  0  0  0  0]
 [ 0  0  1  3  0  0  0  3  0  0  0  0  0  0  0]
 [ 0 16  0  0 13  0  0  4  0  0  1  0  0  0  0]
 [ 0  1  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  7  0  0  3  0  0  0  0  0  0  0  0  0  0]
 [ 0  8  0  0  3  0  2  4  0  0  0  0  0  0  0]
 [ 0  1  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  3  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 0  7  0

In [8]:
# Export classification reports to CSV
report_dfs = {}

for i, lang in enumerate(y_dev.columns):
    true_vals = y_dev[lang]
    pred_vals = y_pred[:, i]
    report_dict = classification_report(true_vals, pred_vals, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).transpose()
    report_dfs[lang] = report_df
    report_df.to_csv(f"{lang}_rfc_classification_report.csv")

print("Classification reports exported.")

Classification reports exported.


In [9]:
from sklearn.metrics import confusion_matrix

for i, lang in enumerate(y_dev.columns):
    true_vals = y_dev[lang]
    pred_vals = y_pred[:, i]
    cm = confusion_matrix(true_vals, pred_vals)
    cm_df = pd.DataFrame(cm, 
                         index=[f"True_{label}" for label in sorted(true_vals.unique())], 
                         columns=[f"Pred_{label}" for label in sorted(true_vals.unique())])
    cm_df.to_csv(f"{lang}_rfc_confusion_matrix.csv")

print("Confusion matrices exported.")

Confusion matrices exported.


In [10]:
#Export features and predictions [note: features only required once; pred is distinct for each experiment]
#X_dev.to_csv("X_dev.csv")
#y_dev.to_csv("y_dev.csv")
pd.DataFrame(y_pred, columns=y_dev.columns).to_csv("y_pred_rfc.csv")

In [ ]:
#Code from p1 working script
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# === Step 1a: build model-facing copies without TokenID ===
X_train_model = X_train.drop(columns=["TokenID"])
X_dev_model   = X_dev.drop(columns=["TokenID"])

# Train the classifier
clf = RandomForestClassifier(random_state=42)
for lang in ["eng_gram", "deu_gram", "cmn_gram"]:
    print(f"\n=== Classification Report for {lang.upper()} ===")

    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_train_model, y_train_bin[lang])   # ← use binarized labels here
    preds = clf.predict(X_dev_model)

    # Convert predictions back into label lists
    pred_labels = mlbs[lang].inverse_transform(preds)
    true_labels = y_dev[lang]

    # Exclude n_a rows
    mask = [lbl != [] for lbl in true_labels]
    true_labels = [lbl for lbl in true_labels if lbl != []]
    pred_labels = [lbl for (lbl, m) in zip(pred_labels, mask) if m]

    print("Sample true:", true_labels[:5])
    print("Sample pred:", pred_labels[:5])

# Predict on dev set (make the name explicit)
rfc_pred = clf.predict(X_dev_model)   # <-- was y_pred

# Align predictions with y_dev for easy filtering later
rfc_pred_df = pd.DataFrame(rfc_pred, index=y_dev.index, columns=y_dev.columns)

# Loop through each language column
for i, lang in enumerate(y_dev.columns):
    print(f"\n=== Classification Report for {lang} ===")
    
    # Extract true values
    true_vals = y_dev[lang]

    # Exclude 'n_a' rows for reporting only
    h = ~true_vals.isin(['n_a'])
    X_dev_loc = X_dev_model[h]
    true_vals = true_vals[h]

    # Predictions for this language column
    pred_vals = rfc_pred_df.loc[h, lang]

    print("Shapes → true_vals:", true_vals.shape, "pred_vals:", pred_vals.shape)
    print("First 10 true:", true_vals.head(10).tolist())
    print("First 10 pred:", pred_vals[:10].tolist())

    # 🔍 Sanity checks (must come *after* pred_vals is created)
    print("Length true_vals:", len(true_vals))
    print("Length pred_vals:", len(pred_vals))
    print("Number equal:", (true_vals == pred_vals).sum())
    print("Manual accuracy:", (true_vals == pred_vals).mean())
    
    # sklearn report
    print(classification_report(true_vals, pred_vals, zero_division=0))